# Zero-shot + Llama SEA-LION v3 8B
Run the reproducible 500-sample stratified test subset.

In [10]:
%pip install -q "vllm>=0.8.3" "bitsandbytes>=0.45.3" "jsonschema>=4.0" rouge-score nltk

Note: you may need to restart the kernel to use updated packages.


In [11]:
MODEL_ID = "aisingapore/Llama-SEA-LION-v3-8B-IT"
SOURCE_MODE = "git"
KAGGLE_REPO_ROOT = "/kaggle/input/poma-repo/POMA"  # only used for SOURCE_MODE=dataset
REPO_URL = "https://github.com/NgDinhKhoi0709/POMA.git"
REVISION = "main"
OUTPUT_ROOT = "/kaggle/working/poma_sea_lion"


In [12]:
import os, subprocess, sys
from pathlib import Path
repo = Path(KAGGLE_REPO_ROOT)
if SOURCE_MODE == "git":
    repo = Path("/kaggle/working/POMA")
    if (repo / ".git").exists():
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", REVISION], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "--detach", "FETCH_HEAD"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REVISION, REPO_URL, str(repo)], check=True)
assert all((repo / "dataset" / name).exists() for name in ["qas_dev.json", "qas_test.json", "table.json"])
os.environ.update({"POMA_LLM_MODEL": "local/sea-lion-v3-8b-it", "POMA_LOCAL_MODEL_ID": MODEL_ID, "POMA_LOCAL_BACKEND": "vllm", "POMA_PROMPT_PROFILE": "compact", "POMA_USE_AGENT_HINTS": "true", "POMA_PARALLEL_WORKERS": "1", "POMA_VLLM_GPU_MEMORY_UTILIZATION": "0.90"})


From https://github.com/NgDinhKhoi0709/POMA
 * branch            main       -> FETCH_HEAD
HEAD is now at 9258d44 fix: reuse existing kaggle repository clone


In [15]:
command = [sys.executable, "scripts/run_sea_lion_kaggle_eval.py", "--repo-root", str(repo), "--output-root", OUTPUT_ROOT, "--phase", "test500", "--mode", "zero_shot", "--model", "local/sea-lion-v3-8b-it"]
subprocess.run(command, cwd=repo, check=True)


POMA:   0%|          | 0/200 [00:00<?, ?qa/s]15:19:28  numexpr.utils                 INFO   NumExpr defaulting to 4 threads.
15:19:29  httpx                         INFO   HTTP Request: HEAD https://huggingface.co/aisingapore/Llama-SEA-LION-v3-8B-IT/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
15:19:29  httpx                         INFO   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/aisingapore/Llama-SEA-LION-v3-8B-IT/fb27c6bafd3fb769a56e904a1e2cf6075fd7240f/config.json "HTTP/1.1 200 OK"
15:19:30  httpx                         INFO   HTTP Request: HEAD https://huggingface.co/aisingapore/Llama-SEA-LION-v3-8B-IT/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
15:19:30  huggingface_hub.utils._http   WARNING  Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
15:19:30  httpx                         INFO   HTTP Request: HEAD https://huggingface.co/

KeyboardInterrupt: 

In [14]:
import shutil
results_dir = Path(OUTPUT_ROOT)
archive_path = shutil.make_archive("/kaggle/working/poma_sea_lion_results", "zip", results_dir)
print(f"Saved results archive: {archive_path}")


Saved results archive: /kaggle/working/poma_sea_lion_results.zip
